In [1]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from engression.models import StoNet
from engression.loss_func import energy_loss_two_sample
from scipy.stats import gamma

ModuleNotFoundError: No module named 'engression.models'

In [ ]:
shape = 2.
scale = 3.
gamma_dist = torch.distributions.Gamma(concentration=shape, rate=1.0/scale)

In [ ]:
# generate data
n = 1000
x = torch.Tensor(gamma_dist.sample((n,1)))

In [ ]:
plt.hist(x.squeeze(1), bins=30, density=True, alpha=0.6, label='Histogram')

# Generate a range of values for plotting the theoretical density curve
xx = np.linspace(min(x), max(x), 1000)
pdf = gamma.pdf(xx, a=shape, scale=scale)

# Overlay the theoretical gamma density curve
plt.plot(xx, pdf, 'r-', lw=2, label='true density')
plt.legend(); plt.show()

In [ ]:
# build model
model = StoNet(in_dim=0, out_dim=1, num_layer=4, hidden_dim=400, noise_dim=1, add_bn=False)#, out_act='softplus')
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# training
model.train()
for i in range(1000):
    optimizer.zero_grad()
    sample1 = model(n)
    sample2 = model(n)
    loss, s1, s2 = energy_loss_two_sample(x, sample1, sample2)
    loss.backward()
    optimizer.step()
    if i == 0 or (i + 1) % 100 == 0:
        print(f'Iter {i + 1} | loss: {loss.item():.4f}, s1: {s1.item():.4f}, s2: {s2.item():.4f}')

In [ ]:
# evaluation
model.eval()
with torch.no_grad():
    sample = model(n)
sum(sample < 0)

In [ ]:
plt.hist(sample.squeeze(1), bins=30, density=True, alpha=0.6, label='fitted')
plt.hist(x.squeeze(1), bins=30, density=True, alpha=0.6, label='training')
xx = np.linspace(min(x), max(x), 1000)
pdf = gamma.pdf(xx, a=shape, scale=scale)
# Overlay the theoretical gamma density curve
plt.plot(xx, pdf, 'r-', lw=2, label='true density')
plt.legend(); plt.show()

In [ ]:
sample1 = model(n)
sample2 = model(n)
loss, s1, s2 = energy_loss_two_sample(x, sample1, sample2)

In [ ]:
loss

In [ ]:
n = 5000

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from scipy.stats import norm # For plotting true density

# --- 1. Define 1D GMM Parameters (using sklearn expected shapes) ---
weights = np.array([0.3, 0.5, 0.2])
n_components = len(weights)
n_features = 1 # Explicitly 1 dimension

# Means (shape: n_components x n_features)
means_sklearn = np.array([[-5.0], [0.0], [8.0]])

# Standard Deviations
std_devs = np.array([1.0, 1.5, 2.0])
# Variances (shape: n_components x n_features x n_features)
variances_sklearn = (std_devs**2).reshape(-1, 1, 1) # Shape (3, 1, 1)
# Precisions (inverse variances) (shape: n_components x n_features x n_features)
precisions_sklearn = (1.0 / (std_devs**2)).reshape(-1, 1, 1) # Shape (3, 1, 1)

n_samples = n

# --- 2. Create and Configure a GaussianMixture object ---
gmm = GaussianMixture(
    n_components=n_components,
    covariance_type='full', # 'full', 'diag', or 'spherical' work for 1D
                           # 'diag' or 'spherical' might feel more natural here
    weights_init=weights,
    means_init=means_sklearn,
    precisions_init=precisions_sklearn
)

# Manually set fitted parameters (as we didn't call fit())
gmm.weights_ = weights
gmm.means_ = means_sklearn
# Need to set covariances_ as well, even if precisions_init was used
gmm.covariances_ = variances_sklearn
# For internal consistency, also set precisions_cholesky_
# In 1D, cholesky(precision) is sqrt(precision) = 1/std_dev
gmm.precisions_cholesky_ = (1.0 / std_devs).reshape(-1, 1, 1)

# --- 3. Generate Samples using the .sample() method ---
# Output shape will be (n_samples, n_features) -> (n_samples, 1)
samples_sklearn, component_labels = gmm.sample(n_samples)

# Flatten samples for easier plotting/analysis in 1D
samples_sklearn_flat = samples_sklearn.flatten()

print(f"Generated {n_samples} 1D samples using sklearn with shape: {samples_sklearn.shape}")
print(f"Flattened shape for plotting: {samples_sklearn_flat.shape}")

# --- 4. Visualize the Samples (Histogram) ---
plt.figure(figsize=(10, 6))
plt.hist(samples_sklearn_flat, bins=60, density=True, alpha=0.7, label='Sampled Distribution (Sklearn)')

# Plot the true density curve (using original means/std_devs for clarity)
x_plot = np.linspace(samples_sklearn_flat.min(), samples_sklearn_flat.max(), 500)
true_density = np.zeros_like(x_plot)
means_1d = means_sklearn.flatten() # Get back to 1D array for norm.pdf
std_devs_1d = np.sqrt(variances_sklearn.reshape(-1)) # Get back to 1D array

for k in range(n_components):
    true_density += weights[k] * norm.pdf(x_plot, loc=means_1d[k], scale=std_devs_1d[k])

plt.plot(x_plot, true_density, 'r-', label='True GMM Density')
plt.title(f'Histogram of IID Samples from Sklearn 1D GMM ({n_components} Components)')
plt.xlabel('Value')
plt.ylabel('Density')
plt.grid(True, alpha=0.5)
plt.legend()
plt.show()

In [ ]:
y = torch.Tensor(samples_sklearn)
# build model
model = StoNet(in_dim=0, out_dim=1, num_layer=4, hidden_dim=100, noise_dim=1, add_bn=False)#, out_act='softplus')
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# training
model.train()
for i in range(1000):
    optimizer.zero_grad()
    sample1 = model(n)
    sample2 = model(n)
    loss, s1, s2 = energy_loss_two_sample(y, sample1, sample2)
    loss.backward()
    optimizer.step()
    if i == 0 or (i + 1) % 100 == 0:
        print(f'Iter {i + 1} | loss: {loss.item():.4f}, s1: {s1.item():.4f}, s2: {s2.item():.4f}')
# evaluation
model.eval()
with torch.no_grad():
    sample = model(n)

In [ ]:
plt.hist(sample.squeeze(1), bins=30, density=True, alpha=0.6, label='fitted')
plt.hist(y.squeeze(1), bins=30, density=True, alpha=0.6, label='training')
xx = np.linspace(min(x), max(x), 1000)
pdf = gamma.pdf(xx, a=shape, scale=scale)
plt.plot(x_plot, true_density, 'r-', label='True GMM Density')
plt.legend(); plt.show()

In [ ]:
y = torch.Tensor(samples_sklearn)
# build model
model = StoNet(in_dim=0, out_dim=1, num_layer=4, hidden_dim=100, noise_dim=1, add_bn=False)#, out_act='softplus')
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# training
model.train()
for i in range(1000):
    optimizer.zero_grad()
    sample1 = model(n)
    sample2 = model(n)
    loss, s1, s2 = energy_loss_two_sample(y, sample1, sample2, beta = 0.3)
    loss.backward()
    optimizer.step()
    if i == 0 or (i + 1) % 100 == 0:
        print(f'Iter {i + 1} | loss: {loss.item():.4f}, s1: {s1.item():.4f}, s2: {s2.item():.4f}')
# evaluation
model.eval()
with torch.no_grad():
    sample = model(n)
plt.hist(sample.squeeze(1), bins=30, density=True, alpha=0.6, label='fitted')
plt.hist(y.squeeze(1), bins=30, density=True, alpha=0.6, label='training')
xx = np.linspace(min(x), max(x), 1000)
pdf = gamma.pdf(xx, a=shape, scale=scale)
plt.plot(x_plot, true_density, 'r-', label='True GMM Density')
plt.legend(); plt.show()

In [ ]:
help(StoNet)